# Phase 1 — Build the Customer Support Knowledge Base




## the project folder



In [1]:
from pathlib import Path

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find the project data folder. Open this notebook inside your capstone-rag-agents project."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder exists:", RAW_DIR.exists())
print("Processed data folder exists:", PROCESSED_DIR.exists())


Project root: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents
Raw data folder exists: True
Processed data folder exists: True


## Step 2 — Create the starter FAQ and policy documents




In [2]:
import csv

documents = [
    {"document_id": "FAQ001", "title": "Track an order", "category": "orders", "source": "sample_faq", "content": "Customers can check an order's status from the Orders page after signing in. Open the order to see its current status, estimated delivery date, and tracking link when available. If tracking has not updated for more than 48 hours, contact support with the order number."},
    {"document_id": "FAQ002", "title": "Cancel an order", "category": "orders", "source": "sample_faq", "content": "An order can usually be cancelled while it is still in the Processing status. Open Orders, select the order, and choose Cancel Order if the option is available. Orders that have already shipped cannot normally be cancelled; customers may request a return after delivery according to the returns policy."},
    {"document_id": "POL001", "title": "Shipping and delivery policy", "category": "shipping", "source": "sample_policy", "content": "Standard delivery typically takes 3 to 7 business days after dispatch. Delivery estimates are estimates and can change because of weather, carrier delays, or public holidays. A tracking link is provided when the carrier makes it available. If a package has no tracking movement for 48 hours, customers should contact support."},
    {"document_id": "FAQ003", "title": "Change a delivery address", "category": "shipping", "source": "sample_faq", "content": "Customers should verify their delivery address before placing an order. An address may be editable from the order details while the order is still Processing. Once the parcel has shipped, the support team cannot guarantee that the carrier can redirect it. Never send full payment-card details when requesting an address change."},
    {"document_id": "POL002", "title": "Returns policy", "category": "returns", "source": "sample_policy", "content": "Eligible items may be returned within 30 calendar days of delivery. Items should be unused and returned with their original packaging, unless they arrived damaged or defective. Some final-sale or customized items may not qualify. Customers should start a return from the order details page and follow the instructions provided. The return request must be approved before an item is sent back."},
    {"document_id": "FAQ004", "title": "Request a refund", "category": "refunds", "source": "sample_faq", "content": "After a return is received and inspected, an approved refund is sent to the original payment method. The time it takes for the funds to appear depends on the bank or payment provider and may take 5 to 10 business days. If the refund has not appeared after that period, contact support with the order number and refund confirmation."},
    {"document_id": "FAQ005", "title": "Item arrived damaged", "category": "returns", "source": "sample_faq", "content": "If an item arrives damaged or defective, contact support within 7 days of delivery. Include the order number, a short description of the issue, and clear photos of the item and packaging when possible. Do not include passwords or full card numbers in the message. Support will explain the next steps after reviewing the request."},
    {"document_id": "POL003", "title": "Payment methods", "category": "payments", "source": "sample_policy", "content": "Available payment methods are shown at checkout and may vary by country. A payment can be declined by the bank, card issuer, or payment provider. Customers can confirm that billing details are correct, check with their provider, and try another available method. Support will never ask customers to share a card PIN, password, or one-time passcode."},
    {"document_id": "FAQ006", "title": "Duplicate payment", "category": "payments", "source": "sample_faq", "content": "Sometimes a pending authorization looks like a second charge. Check whether both entries are posted transactions or whether one is still pending. Pending authorizations may disappear according to the bank's processing time. If two completed charges remain, contact support with the order number and transaction dates. Do not send a full card number or security code."},
    {"document_id": "FAQ007", "title": "Reset a password", "category": "account", "source": "sample_faq", "content": "On the sign-in page, choose Forgot Password and enter the email address associated with the account. Use the reset link sent to that email address and choose a new, unique password. Check the spam folder if the email does not arrive. For safety, support agents cannot tell you your current password."},
    {"document_id": "FAQ008", "title": "Update account email", "category": "account", "source": "sample_faq", "content": "If account settings allow email changes, sign in and open Profile or Account Settings. Enter the new email address and complete the verification step sent to that address. If you cannot access the existing account or verification email, contact support for an account-ownership check. Never share your password or one-time passcodes."},
    {"document_id": "POL004", "title": "Privacy and personal information", "category": "privacy", "source": "sample_policy", "content": "Customers should provide only the information needed to resolve their support request. Do not include passwords, full payment-card numbers, security codes, or one-time passcodes in support messages. Account-specific requests may require verification before changes are made. Personal information should be accessed only by authorized staff for legitimate support purposes."},
    {"document_id": "FAQ009", "title": "Contact customer support", "category": "support", "source": "sample_faq", "content": "Customers can contact support through the Help or Contact Us section of the website. Include a clear description of the problem and the relevant order number when applicable. Do not include sensitive authentication or payment information. Response times can vary depending on request volume and issue complexity."},
    {"document_id": "POL005", "title": "Escalation for unresolved issues", "category": "support", "source": "sample_policy", "content": "If a customer reports that an issue remains unresolved after following the standard process, the case should be reviewed by a support specialist. The summary should include the customer's stated problem, relevant order or case reference, steps already attempted, and the outcome so far. Escalate urgent safety, suspected account compromise, or repeated billing issues promptly. Do not promise a resolution time unless an approved service target is available."},
    {"document_id": "FAQ010", "title": "Missing package", "category": "shipping", "source": "sample_faq", "content": "If tracking says Delivered but the package cannot be found, check the delivery location, mailbox, reception desk, and with household members or neighbors. Allow up to one business day for a carrier status correction. If the package is still missing, contact support with the order number and tracking details so the case can be reviewed."},
    {"document_id": "POL006", "title": "Warranty and defective products", "category": "warranty", "source": "sample_policy", "content": "Warranty coverage depends on the product and the terms shown at purchase. Customers should retain the order confirmation and describe the defect when requesting warranty help. Damage caused by misuse or unauthorized modification may not be covered. Support should check the applicable product terms before confirming eligibility, replacement, repair, or refund options."},
    {"document_id": "FAQ011", "title": "Change or remove saved payment details", "category": "payments", "source": "sample_faq", "content": "Where supported, saved payment methods can be managed from Account Settings or during checkout. Customers may need to add another payment method before removing the default method if there is an active order or subscription. For security, payment details should be updated only through the secure account interface, not by sending card information to support."},
    {"document_id": "FAQ012", "title": "Order marked Processing", "category": "orders", "source": "sample_faq", "content": "Processing means the order has been received but has not yet been dispatched. Processing time can vary with product availability, payment verification, and order volume. The order page should be checked for the latest status. If the status has not changed beyond the estimate shown at checkout, contact support with the order number."},
] 

raw_csv_path = RAW_DIR / "customer_support_knowledge_base.csv"
fieldnames = ["document_id", "title", "category", "source", "content"]
with raw_csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(documents)

print(f"Created {len(documents)} sample source documents.")
print("Saved to:", raw_csv_path)
print("First document title:", documents[0]["title"])


Created 18 sample source documents.
Saved to: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\data\raw\customer_support_knowledge_base.csv
First document title: Track an order


## Step 3 — Load the saved source file and check it




In [3]:
import csv

with raw_csv_path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    source_documents = list(reader)

print("Rows loaded:", len(source_documents))
print("Columns:", list(source_documents[0].keys()))
print("Sample title:", source_documents[0]["title"])
print("Sample content preview:", source_documents[0]["content"][:180] + "...")


Rows loaded: 18
Columns: ['document_id', 'title', 'category', 'source', 'content']
Sample title: Track an order
Sample content preview: Customers can check an order's status from the Orders page after signing in. Open the order to see its current status, estimated delivery date, and tracking link when available. If...


## Step 4 — Clean the document text




In [4]:
import re

def clean_text(text):
    """Normalize whitespace while preserving the original meaning."""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_documents = []
for doc in source_documents:
    cleaned_content = clean_text(doc["content"])
    if not cleaned_content:
        continue
    clean_documents.append({
        "document_id": doc["document_id"].strip(),
        "title": clean_text(doc["title"]),
        "category": clean_text(doc["category"]).lower(),
        "source": clean_text(doc["source"]),
        "content": cleaned_content,
    })

print("Source documents:", len(source_documents))
print("Non-empty cleaned documents:", len(clean_documents))
print("Example cleaned text:", clean_documents[0]["content"][:200] + "...")


Source documents: 18
Non-empty cleaned documents: 18
Example cleaned text: Customers can check an order's status from the Orders page after signing in. Open the order to see its current status, estimated delivery date, and tracking link when available. If tracking has not up...


## Step 5 — Split each document into overlapping chunks




In [5]:
CHUNK_SIZE_WORDS = 100
CHUNK_OVERLAP_WORDS = 20

if CHUNK_OVERLAP_WORDS >= CHUNK_SIZE_WORDS:
    raise ValueError("Chunk overlap must be smaller than chunk size.")

def split_into_chunks(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

chunk_records = []
for doc in clean_documents:
    chunks = split_into_chunks(doc["content"])
    for index, chunk_text in enumerate(chunks, start=1):
        chunk_records.append({
            "chunk_id": f"{doc['document_id']}_CHUNK{index:03d}",
            "document_id": doc["document_id"],
            "title": doc["title"],
            "category": doc["category"],
            "source": doc["source"],
            "chunk_index": index,
            "text": chunk_text,
            "word_count": len(chunk_text.split()),
        })

print("Clean documents:", len(clean_documents))
print("Chunks created:", len(chunk_records))
print("Chunk size setting (words):", CHUNK_SIZE_WORDS)
print("Overlap setting (words):", CHUNK_OVERLAP_WORDS)


Clean documents: 18
Chunks created: 18
Chunk size setting (words): 100
Overlap setting (words): 20


## Step 6 — Inspect the chunks




In [6]:
for record in chunk_records[:5]:
    print("-" * 80)
    print("Chunk ID:", record["chunk_id"])
    print("Title:", record["title"])
    print("Category:", record["category"])
    print("Words:", record["word_count"])
    print("Text:", record["text"])


--------------------------------------------------------------------------------
Chunk ID: FAQ001_CHUNK001
Title: Track an order
Category: orders
Words: 45
Text: Customers can check an order's status from the Orders page after signing in. Open the order to see its current status, estimated delivery date, and tracking link when available. If tracking has not updated for more than 48 hours, contact support with the order number.
--------------------------------------------------------------------------------
Chunk ID: FAQ002_CHUNK001
Title: Cancel an order
Category: orders
Words: 49
Text: An order can usually be cancelled while it is still in the Processing status. Open Orders, select the order, and choose Cancel Order if the option is available. Orders that have already shipped cannot normally be cancelled; customers may request a return after delivery according to the returns policy.
--------------------------------------------------------------------------------
Chunk ID: POL001_CHUNK

## Step 7 — Save processed chunks




In [7]:
import json

jsonl_path = PROCESSED_DIR / "knowledge_base_chunks.jsonl"
chunks_csv_path = PROCESSED_DIR / "knowledge_base_chunks.csv"

with jsonl_path.open("w", encoding="utf-8") as file:
    for record in chunk_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

chunk_fields = ["chunk_id", "document_id", "title", "category", "source", "chunk_index", "text", "word_count"]
with chunks_csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=chunk_fields)
    writer.writeheader()
    writer.writerows(chunk_records)

print("Saved JSONL chunks to:", jsonl_path)
print("Saved CSV chunks to:", chunks_csv_path)
print("JSONL file size (bytes):", jsonl_path.stat().st_size)
print("CSV file size (bytes):", chunks_csv_path.stat().st_size)


Saved JSONL chunks to: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\data\processed\knowledge_base_chunks.jsonl
Saved CSV chunks to: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\capstone-rag-agents\data\processed\knowledge_base_chunks.csv
JSONL file size (bytes): 9564
CSV file size (bytes): 7582


## Step 8 — Validate the saved output



In [8]:
with jsonl_path.open("r", encoding="utf-8") as file:
    saved_chunks = [json.loads(line) for line in file if line.strip()]

required_fields = {"chunk_id", "document_id", "title", "category", "source", "text"}
missing_fields = [
    record["chunk_id"] for record in saved_chunks
    if not required_fields.issubset(record.keys())
]
chunk_ids = [record["chunk_id"] for record in saved_chunks]
duplicate_ids = sorted({chunk_id for chunk_id in chunk_ids if chunk_ids.count(chunk_id) > 1})
empty_text_chunks = [record["chunk_id"] for record in saved_chunks if not record["text"].strip()]

print("Saved chunks reloaded:", len(saved_chunks))
print("Unique chunk IDs:", len(set(chunk_ids)))
print("Missing required fields:", missing_fields)
print("Duplicate chunk IDs:", duplicate_ids)
print("Chunks with empty text:", empty_text_chunks)

assert saved_chunks, "No chunks were saved."
assert not missing_fields, "Some chunks are missing required fields."
assert not duplicate_ids, "Duplicate chunk IDs were found."
assert not empty_text_chunks, "Some chunks have empty text."
print("\nPHASE 1 SUCCESS: source data, cleaned chunks, metadata, and output files are ready.")


Saved chunks reloaded: 18
Unique chunk IDs: 18
Missing required fields: []
Duplicate chunk IDs: []
Chunks with empty text: []

PHASE 1 SUCCESS: source data, cleaned chunks, metadata, and output files are ready.
